In [3]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.abspath(".."))
from model import EEGVAE
from eeg_preprocessing.loaders import build_loaders
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

DATASETS = ["BCI2A"]
RUN_RAW = True
RUN_FEATURES = True
BACKBONE_CKPT = "/home/infres/yrothlin-24/EEG_Foundation_Model/outputs/downstream/tests/downstream_best.pt"

datasets = {
    "PHYSIONETMI": {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/PHYSIONETMI/physionetmi.lmdb", "n_classes": 4},
    "SLEEPEDFX":   {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SLEEPEDF/sleepedf.lmdb", "n_classes": 5},
    "SEEDV":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SEEDV/seedv.lmdb", "n_classes": 5},
    "BCI2A":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/BCI2A/bci2a.lmdb", "n_classes": 4},
    "TUAB":        {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/TUAB/tuab.lmdb", "n_classes": 2},
    "FACED":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/FACED/faced.lmdb", "n_classes": 9},
    "KARAONE":     {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/KARAONE/karaone.lmdb", "n_classes": 11},
    "BCI2020":     {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/BCI2020/bci2020.lmdb", "n_classes": 5},
    "CHISCO":      {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/CHISCO/chisco.lmdb", "n_classes": 39},
    "SIENA":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SIENA/siena.lmdb", "n_classes": 2},
    "SHUMI":       {"dataset_path": "/projects/EEG-foundation-model/RECH202/tests_2/SHUMI/shumi.lmdb", "n_classes": 2},
}

data_config = {
    "batch_size": 64,
    "split_ratio": [0.8, 0.1, 0.1],
    "num_workers": 4,
    "pin_memory": True,
    "persistent_workers": False,
    "seed": 100,
    "channel_mode": "mapped",
    "return_ch_names": False,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(100)
print("device:", device, "| datasets:", DATASETS)


device: cuda | datasets: ['BCI2A']


In [4]:
import torch
from model import EEGVAE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_backbone(ckpt_path):
    config = {
        "in_channels": 19, "out_channels": 19, "adaptor_layers": 4,
        "z_channels": 64, "ch": 64, "ch_mult": [1, 2, 4, 8], "n_blocks": 2,
        "dropout": 0.0, "mode": "vq", "num_embeddings": 256, "n_quantizers": 8,
        "commitment_cost": 0.25, "decay": 0.99,
    }
    sd = torch.load(ckpt_path, map_location="cpu")["model"]
    q = [k for k in sd
         if k.startswith("autoencoder.posterior.quantize.quantizers.") and k.endswith(".embedding")]
    config["z_channels"] = sd[q[0]].shape[1]
    config["n_quantizers"] = len({int(k.split(".quantizers.")[1].split(".")[0]) for k in q})
    model = EEGVAE(**config).to(device).eval()
    model.load_state_dict(sd)
    for p in model.parameters():
        p.requires_grad = False
    return model


@torch.no_grad()
def extract_features(loader):
    feats, labels = [], []
    for x, y in loader:
        feats.append(model.encode(x.to(device).float()).mode().cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)


def feature_loaders(train_loader, val_loader, test_loader):
    B = data_config["batch_size"]
    ztr, ytr = extract_features(train_loader)
    zva, yva = extract_features(val_loader)
    zte, yte = extract_features(test_loader)
    loaders = (
        DataLoader(TensorDataset(ztr, ytr), batch_size=B, shuffle=True),
        DataLoader(TensorDataset(zva, yva), batch_size=B),
        DataLoader(TensorDataset(zte, yte), batch_size=B),
    )
    return loaders, ztr.shape[1], ztr.shape[2]


FEAT_KW = dict(kernel_length=16, depthwise_kernel_length=8,
               pool1_kernel_size=2, pool2_kernel_size=4)

model = load_backbone(BACKBONE_CKPT) if RUN_FEATURES else None

RuntimeError: Error(s) in loading state_dict for EEGVAE:
	Missing key(s) in state_dict: "autoencoder.decoder.conv_in.weight", "autoencoder.decoder.conv_in.bias", "autoencoder.decoder.mid.conv_block.0.depthwise.weight", "autoencoder.decoder.mid.conv_block.0.pointwise.weight", "autoencoder.decoder.mid.conv_block.0.norm.norm.weight", "autoencoder.decoder.mid.conv_block.0.norm.norm.bias", "autoencoder.decoder.mid.conv_block.1.depthwise.weight", "autoencoder.decoder.mid.conv_block.1.pointwise.weight", "autoencoder.decoder.mid.conv_block.1.norm.norm.weight", "autoencoder.decoder.mid.conv_block.1.norm.norm.bias", "autoencoder.decoder.mid.mamba_layer.mamba_forward.dt_bias", "autoencoder.decoder.mid.mamba_layer.mamba_forward.A_log", "autoencoder.decoder.mid.mamba_layer.mamba_forward.D", "autoencoder.decoder.mid.mamba_layer.mamba_forward.in_proj.weight", "autoencoder.decoder.mid.mamba_layer.mamba_forward.conv1d.weight", "autoencoder.decoder.mid.mamba_layer.mamba_forward.conv1d.bias", "autoencode

In [ ]:
from pathlib import Path
import numpy as np
import torch
from pathlib import Path
from sklearn.manifold import TSNE
from eeg_preprocessing.loaders import CustomDataset, get_subject_ids
from torch.utils.data import DataLoader, Dataset

@torch.no_grad()
def extract_codebook(model, lmdb_path, device="cuda", channel_mode="mapped"):
    model.eval()
    embs, labels, subjects = [], [], []
    for sid in get_subject_ids(Path(lmdb_path)):
        ds = CustomDataset(lmdb_path, subject_ids=[sid],
                                channel_mode=channel_mode)
        loader = DataLoader(ds, batch_size=256, num_workers=4)
        for x, y in loader:
            z = model.encode(x.to(device).float()).mode().mean(dim=-1)
            embs.append(z.cpu().numpy())
            labels.append(y.numpy())
            subjects.append(np.full(len(y), sid))
    return np.concatenate(embs), np.concatenate(labels), np.concatenate(subjects)

LMDB = datasets["BCI2A"]["dataset_path"]

emb, y, subj = extract_codebook(model, LMDB)
proj = TSNE(n_components=2, init="pca", perplexity=30).fit_transform(emb)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for sid in np.unique(subj):
    m = subj == sid
    axes[0].scatter(proj[m, 0], proj[m, 1], s=6, alpha=0.6, label=str(sid))
axes[0].set_title("t-SNE codebook features by subject")
axes[0].legend(markerscale=2, fontsize=8, loc="best")
for c in np.unique(y):
    m = y == c
    axes[1].scatter(proj[m, 0], proj[m, 1], s=6, alpha=0.6, label=str(c))
axes[1].set_title("t-SNE codebook features by class")
axes[1].legend(markerscale=2, fontsize=8, loc="best")
plt.tight_layout()
plt.show()
